# Kelly Rule

Modernized, dependency-light version for Python 3.10+.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 80)
FOLDER = Path.cwd().resolve()
if str(FOLDER) not in sys.path:
    sys.path.insert(0, str(FOLDER))
from strategy_evaluation_utils import (
    make_price_panel, returns_matrix, moving_average_signal, equal_weight_returns,
    performance_table, mean_variance_weights, portfolio_returns, kelly_fraction, export_strategy_outputs
)
OUTPUT = FOLDER.parent / 'data' / 'strategy_evaluation'
OUTPUT.mkdir(parents=True, exist_ok=True)
prices = make_price_panel()
rets = returns_matrix(prices)


In [2]:
strategy = moving_average_signal(prices).groupby('date')['strategy_return'].mean().dropna()
wins = strategy[strategy > 0]
losses = strategy[strategy < 0]
win_probability = float((strategy > 0).mean())
win_loss_ratio = float(wins.mean() / abs(losses.mean())) if not losses.empty else 0.0
kelly = kelly_fraction(win_probability, win_loss_ratio)
scaled = (strategy * kelly).rename('kelly_scaled')
summary = pd.DataFrame([{'win_probability': win_probability, 'win_loss_ratio': win_loss_ratio, 'kelly_fraction': kelly}])
perf = performance_table({'raw_strategy': strategy, 'kelly_scaled': scaled})
export_strategy_outputs(OUTPUT, kelly_summary=summary, kelly_scaled_returns=scaled)
display(summary)
display(perf)

,win_probability,win_loss_ratio,kelly_fraction
0,0.515873,0.979756,0.021743


,strategy,annual_return,annual_volatility,sharpe,max_drawdown,hit_rate
0,raw_strategy,0.101225,0.073000,1.386637,-0.068050,0.515873
1,kelly_scaled,0.002099,0.001587,1.322493,-0.001525,0.515873
